## Install BFCL and vLLM.

In [1]:
import os

!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.11.0', 'triton==3.2.0') if is_t4 else ('vllm==vllm==0.11.0', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.57.6
!uv pip install --no-deps trl==0.22.2
!uv pip uninstall flashinfer-python

os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

!git clone https://github.com/ShishirPatil/gorilla.git
%cd gorilla/berkeley-function-call-leaderboard
!uv pip install -e .[oss_eval_vllm]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.0 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 18 packages in 218ms
Prepared 2 packages in 640ms
Uninstalled 2 packages in 79ms
Installed 2 packages in 59ms
 - huggingface-hub==1.14.0
 + huggingface-hub==0.36.2
 - transformers==5.8.0
 + transformers==4.57.6
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 86ms
Prepared 1 package in 61ms
Installed 1 package in 5ms
 + trl==0.22.2
Using Python 3.12.13 environment at: /usr
Cloning into 'gorilla'...
remote: Enumerating objects: 6789, done.
remote: Total 6789 (delta 0), reused 0 (delta 0), pack-reused 6789 (from 2)
Receiving objects: 100% (6789/6789), 371.22 MiB | 35.01 MiB/s, done.
Resolving deltas: 100% (4750/4750), done.
/content/gorilla/berkeley-function-call-leaderboard
Using Python 3.12.13 environment at: /usr
Resolved 210 packages in 4.17s
Prepared 78 packages in 1m 02s
Uninstalled 39 packages in 637ms
Installed 78 packages in 626ms
 + ai

## Add model config to file.

In [2]:
model_name = "unsloth/Phi-4-mini-instruct"
adapter_name = "jeypiii/Phi-4-mini-instruct_Long_CoT_v3".lower()

config_name = adapter_name.replace("_", "-")
display_name = adapter_name.split("/")[1]

extra_config = f"""
from bfcl_eval.model_handler.local_inference.agshort import AgShortHandler

MODEL_CONFIG_MAPPING = {{
    **api_inference_model_map,
    **local_inference_model_map,
    **third_party_inference_model_map,
    '{config_name}': ModelConfig(
        model_name    = '{adapter_name}',
        display_name  = '{display_name} (Prompt)',
        url           = 'https://huggingface.co/{adapter_name}',
        org           = 'Ubiquitous Computing Laboratory',
        license       = 'MIT',
        model_handler = AgShortHandler,
        is_fc_model   = False,
    )
}}
"""

with open("bfcl_eval/constants/model_config.py", "a") as f:
  f.write(extra_config)

## Define and write model handler.

In [3]:
%%writefile bfcl_eval/model_handler/local_inference/agshort.py
import json
from typing import Any
from bfcl_eval.model_handler.local_inference.base_oss_handler import OSSHandler
from overrides import override
import time

class AgShortHandler(OSSHandler):
    def __init__(
        self,
        model_name,
        temperature,
        registry_name,
        is_fc_model,
        dtype="bfloat16",
        **kwargs,
    ) -> None:
        super().__init__(model_name, temperature, registry_name, is_fc_model, **kwargs)
        self.stop_token_ids = [200020, 199999]

    @staticmethod
    def _tools_formatting_func(schema):
        IGNORED_KEYS = {"type", "description", "required", "properties", "items", "default", "enum", "optional"}
        def format_dict(d, required=None, indent=""):
            required = set(required or [])
            lines = []
            for key, val in d.items():
                type_str = val.get("type", "unknown")
                if key not in required:
                    type_str += ", optional"
                desc = build_description(val, indent)
                lines.append(f"{indent}- {key} ({type_str}): {desc}")
            return "\n".join(lines)
        def build_description(val, indent):
            desc = val.get("description", "").rstrip(".") + "."
            if "default" in val and "default" not in desc.lower():
                desc += f" Default is {repr(val['default'])}."
            if "enum" in val:
                enum_vals = ", ".join(map(repr, val["enum"]))
                desc += f" Possible values are {enum_vals}."
            # Nested dict
            if val.get("type", "").startswith("dict") and "properties" in val:
                nested = format_dict(val["properties"], val.get("required"), indent + "  ")
                desc += " Properties:\n" + nested
            # Items handling
            if "items" in val:
                desc += " For each item:\n" + format_items(val["items"], indent + "  ")
            # Extra fields
            extras = [
                f"{indent}  - {k.capitalize()}: {repr(v)}"
                for k, v in val.items()
                if k not in IGNORED_KEYS
            ]
            if extras:
                desc += "\n" + "\n".join(extras)
            return desc
        def format_items(item, indent):
            lines = []
            if "type" in item:
                lines.append(f"{indent}- Type: {item['type']}")
            if "enum" in item:
                enum_vals = ", ".join(map(repr, item["enum"]))
                lines.append(f"{indent}- Possible values: {enum_vals}.")
            if "properties" in item:
                lines.append(f"{indent}- Properties:")
                lines.append(format_dict(item["properties"], item.get("required"), indent + "  "))
            for k, v in item.items():
                if k not in {"type", "enum", "properties"}:
                    lines.append(f"{indent}- {k.capitalize()}: {repr(v)}")
            return "\n".join(lines)
        result = []
        for func in schema:
            name = func.get("name", "unknown")
            desc = func.get("description", "").rstrip(".") + "." + (" Parameters:" if func.get("parameters", {}) != {} else "")
            result.append(f"- {name}: {desc}")
            params = func.get("parameters", {})
            if params.get("properties"):
                result.append(format_dict(params["properties"], params.get("required"), "  "))
            for k, v in params.items():
                if k not in {"type", "properties", "required"}:
                    result.append(f"  - {k.capitalize()}: {repr(v)}")
        return "\n".join(result)

    @override
    def _pre_query_processing_prompting(self, test_entry: dict) -> dict:
        functions: list = test_entry["function"]
        return {"message": [], "function": functions}

    @override
    def _format_prompt(self, messages, function):
        system_message = ""
        remaining_messages = messages
        if messages[0]["role"] == "system":
            system_message = messages[0]["content"].strip()
            remaining_messages = messages[1:]

        formatted_prompt =  "<|system|>"
        formatted_prompt += "You are a fast and responsive reasoning assistant.\n"
        formatted_prompt += "Your job is to answer questions and instructions with a concise and to-the-point thought process."
        formatted_prompt += f"\n\n{system_message}" if (system_message != "") else ""
        formatted_prompt += "<|end|>"

        is_first_user_message = True
        for message in remaining_messages:
            if message["role"] == "user" and is_first_user_message:
                is_first_user_message = False
                formatted_prompt += "<|user|>"
                formatted_prompt += f"{message['content'].strip()}\n"
                formatted_prompt += "Here is a list of functions that you can invoke:\n"
                formatted_prompt += self._tools_formatting_func(function) + "\n"
                formatted_prompt += "Respond only with a list of function calls in Python format. No explanations or any other text. Example: [func1(x=1, y=2, ...), func2(), ...]\n"
                formatted_prompt += "If there are no appropriate functions to help the task, simply respond with an empty list. Example: []"
                formatted_prompt += "<|end|>"

            elif message["role"] == "assistant":
                content = message["content"]
                content = content.split(".answer")[-1].strip()
                formatted_prompt += f"<|assistant|>{content}<|end|>"

            elif message["role"] == "tool":
                formatted_prompt += "<|user|><tool_response>\n"
                if isinstance(message["content"], (dict, list)):
                    formatted_prompt += json.dumps(message["content"])
                else:
                    formatted_prompt += message["content"]
                formatted_prompt += "\n</tool_response><|end|>"

            else:
                formatted_prompt += f"<|{message['role']}|>{message['content'].strip()}<|end|>"

        formatted_prompt += "<|assistant|>"

        return formatted_prompt

    @override
    def _parse_query_response_prompting(self, api_response: Any) -> dict:
        model_response = api_response.choices[0].text
        reasoning_content = ""
        if ".answer" in model_response:
            reasoning_content = model_response.split(".answer")[0].strip()
            model_response = model_response.split(".answer")[-1].strip()

        return {
            "model_responses": model_response,
            "reasoning_content": reasoning_content,
            "model_responses_message_for_chat_history": api_response.choices[0].text,
            "input_token": api_response.usage.prompt_tokens,
            "output_token": api_response.usage.completion_tokens,
        }

    @override
    def _add_assistant_message_prompting(
        self, inference_data: dict, model_response_data: dict
    ) -> dict:
        inference_data["message"].append(
            {
                "role": "assistant",
                "content": model_response_data["model_responses_message_for_chat_history"],
            }
        )
        return inference_data

    @override
    def _query_prompting(self, inference_data: dict):
        # We use the OpenAI Completions API
        function: list[dict] = inference_data["function"]
        message: list[dict] = inference_data["message"]

        formatted_prompt: str = self._format_prompt(message, function)
        inference_data["inference_input_log"] = {"formatted_prompt": formatted_prompt}

        # Tokenize the formatted prompt to get token count
        input_token_count = len(self.tokenizer.tokenize(formatted_prompt))

        # Determine the number of tokens to request. Cap it at 4096 if the model has a larger limit.
        if self.max_context_length < input_token_count + 2:
            # If the prompt is already at the max length, just request 1000 token, we will get an error anyway
            leftover_tokens_count = 1000
        else:
            leftover_tokens_count = min(
                4096,
                self.max_context_length - input_token_count - 2,
            )

        extra_body = {}
        if hasattr(self, "stop_token_ids"):
            extra_body["stop_token_ids"] = self.stop_token_ids
        if hasattr(self, "skip_special_tokens"):
            extra_body["skip_special_tokens"] = self.skip_special_tokens

        start_time = time.time()
        if len(extra_body) > 0:
            api_response = self.client.completions.create(
                model=self.model_path_or_id,
                prompt=formatted_prompt,
                max_tokens=leftover_tokens_count,
                extra_body=extra_body,
                timeout=72000,  # Avoid timeout errors
                temperature = 0.7,
                top_p = 0.8,
                presence_penalty = 1.5,
                seed = 0,
            )
        else:
            api_response = self.client.completions.create(
                model=self.model_path_or_id,
                prompt=formatted_prompt,
                max_tokens=leftover_tokens_count,
                timeout=72000,  # Avoid timeout errors
                temperature = 0.7,
                top_p = 0.8,
                presence_penalty = 1.5,
                seed = 0,
            )
        end_time = time.time()

        return api_response, end_time - start_time

Writing bfcl_eval/model_handler/local_inference/agshort.py


## Start the vLLM server.

In [4]:
import time
get_ipython().system = os.system   # kaggle background process fix

cmd = f"""\
python -m vllm.entrypoints.openai.api_server \
  --model {model_name} \
  --gpu-memory-utilization 0.8 \
  --max-model-len 8500 \
  --max-num-seqs 32 \
  --port 1053 \
  --dtype float16 \
  --tensor-parallel-size 1 \
  --quantization bitsandbytes \
  --enable-lora \
  --max-lora-rank 16 \
  --lora-modules \
      {adapter_name}={adapter_name} \
  > vllm.log 2>&1 &
"""
os.system(cmd)

print("Waiting for vLLM server to start...")
while True:
  time.sleep(10)
  log_lines = !tail -n 1 vllm.log
  last_line = log_lines[0] if log_lines else ""
  print(f"Last Log Entry: {last_line}")
  if "Application startup complete." in last_line:
    break

Waiting for vLLM server to start...
Last Log Entry: 
Last Log Entry: 
Last Log Entry: INFO 05-08 10:07:14 [__init__.py:239] Automatically detected platform cuda.
Last Log Entry: WARNING 05-08 10:07:24 [config.py:2972] Casting torch.bfloat16 to torch.float16.
Last Log Entry: WARNING 05-08 10:07:24 [config.py:2972] Casting torch.bfloat16 to torch.float16.
Last Log Entry: INFO 05-08 10:07:44 [api_server.py:246] Started engine process with PID 6615
Last Log Entry: INFO 05-08 10:07:52 [__init__.py:239] Automatically detected platform cuda.
Last Log Entry: INFO 05-08 10:08:06 [weight_utils.py:265] Using model weights format ['*.safetensors']
Last Log Entry: INFO 05-08 10:08:06 [weight_utils.py:265] Using model weights format ['*.safetensors']
Last Log Entry: INFO 05-08 10:08:06 [weight_utils.py:265] Using model weights format ['*.safetensors']
Last Log Entry: INFO 05-08 10:08:06 [weight_utils.py:265] Using model weights format ['*.safetensors']
Last Log Entry: INFO 05-08 10:08:06 [weight_uti

## Test server with OpenAI completion.

In [5]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1053/v1", api_key="EMPTY")

prompt = """\
<|system|>\
You are a fast and responsive reasoning assistant.
Your job is to answer questions and instructions with a concise and to-the-point thought process.<|end|>\
<|user|>\
What is the result of dividing 100 by 25?
Here is a list of functions that you can invoke:
- historical_exchange_rates: Fetch historical exchange rate data for a specific date using the RapidAPI service. Parameters:
  - date (string): The date for which to retrieve exchange rate data, formatted as 'YYYY-MM-DD'.
- hull_moving_average: Calculates the Hull Moving Average (HMA) indicator for a given cryptocurrency market symbol using the Qvantana API. Parameters:
  - exchange (string): Name of the cryptocurrency exchange (e.g., 'binance'). Default is 'binance'.
  - market (string): Type of the market (e.g., 'spot', 'usdt-perpetual'). Default is 'spot'.
  - symbol (string): Trading symbol of the cryptocurrency (e.g., 'BTCUSD'). Default is 'btcusdt'.
  - interval (string): Time interval for the HMA calculation (e.g., '1d', '1h'). Default is '1m'.
  - is_from (string, optional): Start time for fetching data in UNIX timestamp format, default is '1683895800'.
  - backtracks (integer, optional): Number of periods to look back, default is 1.
  - length (integer, optional): Length of the HMA, default is 9.
Respond only with a list of function calls in Python format. No explanations or any other text. Example: [func1(x=1, y=2, ...), func2(), ...]
If there are no appropriate functions to help the task, simply respond with an empty list. Example: []<|end|>\
<|assistant|>\
"""

completion = client.completions.create(
  model = adapter_name.lower(),
  prompt = prompt,
  max_tokens = 7000,
  temperature = 0.7,
  top_p = 0.8,
  presence_penalty = 1.5,
  seed = 0,
)

print(completion.choices[0].text)

.reason
Okay, the user wants the result of dividing 100 by 25. That's a straightforward math problem. Let me think. Oh right, 100 divided by 25 equals 4. Simple division.

But wait, the user also provided a list of functions that I can call. Hmm... none of those functions seem directly related to basic arithmetic operations. The historical_exchange_rates function is for fetching exchange rates, hull_moving_average is for calculating financial indicators - these are all about data retrieval and analysis, not simple math.

The task specifically says "Respond only with a list of function calls in Python format." Since none of the given functions can perform a division operation, I should just return an empty list.

Wait, maybe I misunderstood something. Let me double-check the question again. The user asked: "What is the result of dividing 100 by 25?" They want the mathematical answer. But the assistant's instructions say to use the provided functions if possible. None of them can do divi

## Generate BFCL completions.

In [ ]:
cmd = f"""\
bfcl generate \
  --model {config_name} \
  --test-category live \
  --temperature 0.7 \
  --include-input-log \
  --skip-server-setup
"""
os.system(cmd)

In [ ]:
os.system(f"zip -r {display_name}_results.zip result")

## Evaluate completions.

In [ ]:
cmd = f"""\
bfcl evaluate \
  --model {config_name} \
  --test-category live
"""
os.system(cmd)

In [ ]:
os.system(f"zip -r {display_name}_score.zip score")

In [ ]:
import json

print("Output Lengths (tok):")

for root, _, files in os.walk("result"):
  for f in files:
    if not f.endswith(".json"):
      continue
    path = os.path.join(root, f)
    lines = [json.loads(l) for l in open(path) if l.strip()]
    if not lines:
      continue
    avg = sum(d.get("output_token_count", 0) for d in lines) / len(lines)
    print(f">> {'/'.join(path.split(os.sep)[-2:])} ({len(lines)} samples): {avg}")